# Self-Disclosure Detection Pipeline

Runs annotation, training, and the experiment grid on a Colab GPU.

**Runtime > Change runtime type > T4 GPU** before starting.

## Two modes

`DRY_RUN = True` uses an innocuous public text dataset to verify the whole
pipeline end to end and measure throughput. Run this first. It touches no
sensitive data and needs no ethics approval, so it can be done while the
Secondary Data Checklist is still with the supervisor.

`DRY_RUN = False` uses the real corpus. **Do not switch this until the
checklist is signed.**

The point of the dry run is that annotating tens of thousands of posts takes
hours, and discovering a batch size problem or a session timeout at that point
is expensive. Better to find it now on data that does not matter.

## 1. Setup

In [ ]:
!nvidia-smi

# Confirm a GPU is actually attached. Colab will happily give you a CPU
# runtime, and the annotation run would then take about forty times longer.
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets scikit-learn pandas matplotlib tqdm

In [ ]:
# Pull the project code.
REPO = "https://github.com/hsnnaw/dissertation.git"
PROJECT = "/content/project"

import os, sys

# Absolute paths throughout, so re-running this cell after the %cd below
# behaves the same as running it fresh.
if os.path.isdir(PROJECT):
    !cd $PROJECT && git pull --ff-only
else:
    !git clone $REPO $PROJECT

%cd $PROJECT
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
print("ready")


In [ ]:
# Persist outputs to Drive so a session timeout does not lose hours of work.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
WORK = Path("/content/drive/MyDrive/dissertation")
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / "outputs").mkdir(parents=True, exist_ok=True)
print(WORK)

## 2. Mode

The Secondary Data Checklist is signed, so this runs on the real corpus.
Set `DRY_RUN = True` to go back to the news-text rehearsal, which is worth
doing after any change to the annotation path.


In [ ]:
DRY_RUN = False

# Dry run: small enough to iterate on, large enough for timings to
# extrapolate.
#
# Real run: 5,000. At the measured rate on real posts this is roughly four
# hours, which is about what one Colab session reliably gives. 5,000
# labelled posts is ample to fine-tune RoBERTa, and every experiment in the
# grid works at that scale. Annotation appends and resumes on post_id, and
# both the sample and the ids are deterministic, so raising this later
# extends the corpus rather than redoing it.
N_POSTS = 500 if DRY_RUN else 5_000

STRATEGY = "few_shot"          # best on the benchmark: 0.902 at 1.78s/post
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

print(f"DRY_RUN={DRY_RUN}  posts={N_POSTS}  strategy={STRATEGY}")
print(f"model={MODEL_ID}")
if not DRY_RUN:
    print("\nREAL CORPUS. This requires the signed Secondary Data Checklist.")


## 3. Input data

The real corpus is the Low et al. Reddit Mental Health Dataset
([Zenodo 3941387](https://zenodo.org/records/3941387), ODC-PDDL). The full
record is 3.1 GB; the cells below take only the `post` timeframe for the
communities in use, roughly 800 MB, onto Colab's local disk rather than Drive.

In dry run mode this is skipped and a small corpus of neutral news text is
built instead, with the same shape: a `text` field, a `post_id`, and a
`subreddit`.


In [ ]:
# Low et al. Reddit Mental Health Dataset, Zenodo record 3941387.
# https://zenodo.org/records/3941387   ODC Public Domain Dedication & License
#
# Downloaded to Colab's local disk, not Drive. The full record is 3.1 GB and
# only a subset is needed.
import urllib.request
from pathlib import Path

RECORD = "3941387"
BASE = f"https://zenodo.org/records/{RECORD}/files"
RAW = Path("data/raw")

# One timeframe only. Mixing windows would make period a confound in the
# cross-community generalisation experiment. "post" is Jan-Apr 2020.
TIMEFRAME = "post"

# suicidewatch and EDAnonymous are deliberately excluded: crisis content and
# eating disorder content respectively. Thirteen support communities remain,
# which is ample. Record this exclusion and the reasoning in the write-up:
# a justified exclusion is a stronger position than an unexamined inclusion.
MH = ["addiction", "alcoholism", "adhd", "anxiety", "autism",
      "bipolarreddit", "bpd", "depression", "healthanxiety", "lonely",
      "ptsd", "schizophrenia", "socialanxiety", "mentalhealth"]

# Negative examples. Holding out whole communities is only meaningful if the
# held-out set spans both groups, so both are needed.
CONTROL = ["conspiracy", "divorce", "fitness", "guns", "jokes",
           "legaladvice", "meditation", "parenting", "personalfinance",
           "relationships", "teaching"]


def fetch(subreddit: str) -> Path:
    """Download one subreddit file, skipping it if already present."""
    name = f"{subreddit}_{TIMEFRAME}_features_tfidf_256.csv"
    dst = RAW / name
    if dst.exists() and dst.stat().st_size > 0:
        return dst
    # Download to a temporary name and rename on success, so an interrupted
    # download is never mistaken for a complete file on the next run.
    part = dst.with_suffix(".part")
    urllib.request.urlretrieve(f"{BASE}/{name}?download=1", part)
    part.rename(dst)
    return dst


# Probe one small file before committing to the rest. These are named
# "features_tfidf_256", so confirm they carry post text and not only feature
# vectors. Two minutes here beats discovering it after 800 MB.
if DRY_RUN:
    print("dry run: skipping corpus download")
else:
    import pandas as pd

    RAW.mkdir(parents=True, exist_ok=True)
    probe = fetch("addiction")
    head = pd.read_csv(probe, nrows=3, low_memory=False)
    TEXT_CANDIDATES = ("post", "selftext", "body", "text")   # same as src.data
    found = next((c for c in TEXT_CANDIDATES if c in head.columns), None)

    print(f"{probe.name}: {len(head.columns)} columns")
    if found is None:
        raise SystemExit(
            f"No text column in {probe.name}. Looked for {TEXT_CANDIDATES}.\n"
            f"Columns: {list(head.columns)[:20]}\n"
            "Without post text the annotation approach does not apply here."
        )
    print(f"text column: {found!r}")
    print(f"sample: {str(head[found].iloc[0])[:200]}")


In [ ]:
# Only run this once the probe above reports a usable text column.
if DRY_RUN:
    print("dry run: skipping corpus download")
else:
    import time

    started = time.perf_counter()
    for sub in MH + CONTROL:
        try:
            path = fetch(sub)
            print(f"  {path.stat().st_size/1e6:7.1f} MB  {path.name}")
        except Exception as exc:
            print(f"  FAILED  {sub}: {exc}")

    files = sorted(RAW.glob("*.csv"))
    total = sum(f.stat().st_size for f in files)
    print(f"\n{len(files)} files, {total/1e6:.0f} MB, "
          f"{time.perf_counter()-started:.0f}s")

    leftover = list(RAW.glob("*.part"))
    if leftover:
        print(f"\nincomplete, rerun this cell: {[p.name for p in leftover]}")


In [ ]:
import json, random
from pathlib import Path

POSTS = Path("data/interim/posts.jsonl")
POSTS.parent.mkdir(parents=True, exist_ok=True)

if DRY_RUN:
    from datasets import load_dataset

    # AG News: ordinary news text, nothing sensitive. Any neutral corpus of
    # comparable length would do equally well.
    ds = load_dataset("fancyzhx/ag_news", split=f"train[:{N_POSTS}]")
    topics = ["world", "sports", "business", "scitech"]

    with POSTS.open("w") as f:
        for i, row in enumerate(ds):
            f.write(json.dumps({
                "post_id": f"dry{i:05d}",
                "text": row["text"],
                "subreddit": topics[row["label"]],
                "group": "non_mh",
            }) + "\n")
    print(f"dry run corpus: {N_POSTS} posts")

else:
    # Real corpus. Requires the signed checklist.
    #
    # Put the dataset CSVs in data/raw/ first. prepare() reads every CSV in
    # that directory, takes the subreddit and timeframe from each filename,
    # and probes for the text column, which the releases name differently.
    RAW = Path("data/raw")
    csvs = sorted(RAW.glob("*.csv")) if RAW.exists() else []
    if not csvs:
        raise FileNotFoundError(
            f"No CSVs in {RAW.resolve()}. Copy the dataset there before "
            f"running this cell, for example from Drive:\n"
            f"  !mkdir -p {RAW} && cp /content/drive/MyDrive/<your-folder>/*.csv {RAW}/"
        )
    print(f"{len(csvs)} CSVs in {RAW}")

    from src.data import prepare
    prepare(
        input_dir=RAW,
        output_path=POSTS,
        sample=N_POSTS,
    )

print(sum(1 for _ in POSTS.open()), "posts ready")


## 4. Annotation

Runs the annotator locally on the Colab GPU. No text leaves the machine,
which is the condition the ethics route depends on.

Llama 3.1 is licence-gated. Accept the licence on the model page and supply
a read token below, or the model download fails with a gated-repo error.
`Qwen/Qwen2.5-7B-Instruct` is the ungated fallback, and is what the dry run
used.

At the measured 1.485 s/post this is roughly four hours for 10,000 posts.
Annotation is resumable: if the session drops, rerun this cell and it picks
up from where it stopped rather than starting over.

**Watch the confidence spread on the first batch.** Every dry run record
came back at 0.95 or above. If that repeats on real data the confidence
field carries no information, and the agreement study's stratified sample,
which splits at 0.8, collapses into a single band.


In [ ]:
from huggingface_hub import login
login()  # paste a token with read access

In [ ]:
import time
from src.annotate import run

# The labels file lives on Drive, not on Colab's disk. It is the expensive
# artefact -- hours of GPU time -- and a runtime recycle wipes local disk
# without warning, which loses the lot. It carries post_id and labels only,
# no post text, so Drive is an appropriate place for it.
#
# The write rate is one flush per batch, roughly every 35 seconds, which is
# nothing for Drive. Posts and splits stay local: they hold text, and both
# rebuild deterministically from data/raw in under a minute.
LABELS = WORK / f"labels_{STRATEGY}.jsonl"
LABELS.parent.mkdir(parents=True, exist_ok=True)
print(f"labels -> {LABELS}")
if LABELS.exists():
    print(f"  {sum(1 for _ in LABELS.open())} already annotated, will resume")

# Annotation is resumable, so rerunning this cell on a finished file does
# nothing and times nothing. Delete the file to measure a fresh run.
REDO = False
if REDO and LABELS.exists():
    LABELS.unlink()
    print(f"deleted {LABELS}, annotating from scratch")

started = time.perf_counter()
stats = run(
    input_path=POSTS,
    output_path=LABELS,
    strategy=STRATEGY,
    backend_kind="transformers",
    model=MODEL_ID,
    batch_size=16,
)
elapsed = time.perf_counter() - started

n = stats["ok"] + stats["failed"]

if n == 0:
    print("\nNothing to annotate: every post in POSTS is already in LABELS.")
    print("Set REDO = True above to time a fresh run.")
else:
    # Separate the one-off cost from the per-post cost. Downloading and
    # loading the model happens once whether the corpus is 500 posts or
    # 20,000, so extrapolating from wall clock overstates the real run.
    annotation_s = stats["total_latency_s"]
    setup_s = elapsed - annotation_s

    print(f"\n{n} posts in {elapsed/60:.1f} min wall clock")
    print(f"  model download and load : {setup_s/60:5.1f} min  (one-off)")
    print(f"  annotation              : {annotation_s/60:5.1f} min  "
          f"= {annotation_s/n:.3f} s/post")
    # What a larger corpus would cost, if it is worth extending later.
    per_post = annotation_s / n
    for target in (10_000, 20_000):
        print(f"  {target:,} posts would be {per_post*target/3600:.1f} h "
              f"annotation + {setup_s/60:.0f} min setup")
    print(f"parse failure rate: {stats.get('failure_rate', 0):.1%}")

# Token spread reads from the file, so it works whether or not anything was
# annotated just now. Check it before lowering max_tokens: cutting below the
# longest real response truncates valid JSON into a parse failure.
toks = sorted(r["output_tokens"] for r in map(json.loads, LABELS.open())
              if "output_tokens" in r)
if toks:
    p = lambda q: toks[min(int(len(toks) * q), len(toks) - 1)]
    cap = 128  # the few_shot default in src/annotate.py
    print(f"\noutput tokens over {len(toks)} records")
    print(f"  median={p(0.5)}  p90={p(0.9)}  p99={p(0.99)}  max={toks[-1]}")
    print(f"  at or above the {cap}-token cap: {sum(t >= cap for t in toks)}")


**Read the extrapolation before continuing.** If 20,000 posts would take longer
than a Colab session allows, the options are a larger batch size, a smaller
corpus, or splitting the run across sessions. Annotation is resumable, so the
last of those works: rerun the same cell and it picks up where it stopped.

A parse failure rate above a few percent means the prompt needs attention
before committing to the full run.

## 5. Splits

In [ ]:
from src.splits import load_labelled, stratified_split, subreddit_split, summarise

# The annotation output carries labels keyed by post_id but not the post
# text, so join it back from POSTS. Training needs both.
records = load_labelled(LABELS, posts_path=POSTS)
print(f"{len(records)} labelled records")

for mode, splitter in [("random", stratified_split), ("subreddit", subreddit_split)]:
    splits = splitter(records)
    summarise(splits, "is_disclosure")
    outdir = Path(f"data/processed/splits_{mode}")
    outdir.mkdir(parents=True, exist_ok=True)
    for name, group in splits.items():
        with (outdir / f"{name}.jsonl").open("w") as f:
            for r in group:
                f.write(json.dumps(r) + "\n")
    print(f"wrote {outdir}\n")


**Check the class balance before training.** If the positive rate is below
about 5%, the classifier will struggle regardless of class weighting, and the
corpus sampling needs rethinking rather than the model.

In dry run mode expect a very low positive rate, since news text contains no
self-disclosure. That is the correct result: it confirms the annotator is not
firing on everything.

## 6. Training

In [ ]:
from src.train import train

result = train(
    splits_dir=Path("data/processed/splits_random"),
    output_dir=Path("outputs/experiments/gen_seen"),
    model_name="roberta-base",
    epochs=3,
    batch_size=16,
)

## 7. Experiment grid

Skip in dry run mode. The numbers would be meaningless and it costs an hour.

In [ ]:
if not DRY_RUN:
    # Noise robustness
    for rate in [0.0, 0.05, 0.10, 0.20, 0.30]:
        train(
            splits_dir=Path("data/processed/splits_random"),
            output_dir=Path(f"outputs/experiments/noise_{rate}"),
            noise_rate=rate,
        )

    # Generalisation
    train(
        splits_dir=Path("data/processed/splits_subreddit"),
        output_dir=Path("outputs/experiments/gen_unseen"),
    )

    # Size against cost
    for m in ["roberta-base", "distilroberta-base"]:
        train(
            splits_dir=Path("data/processed/splits_random"),
            output_dir=Path(f"outputs/experiments/size_{m}"),
            model_name=m,
        )

    # Class weighting ablation
    train(
        splits_dir=Path("data/processed/splits_random"),
        output_dir=Path("outputs/experiments/no_weights"),
        class_weights=False,
    )
else:
    print("skipped in dry run")

## 8. Analysis

In [ ]:
!python -m scripts.analyse breakdown --dir outputs/experiments/gen_seen --plot

if not DRY_RUN:
    !python -m scripts.analyse noise --dir outputs/experiments
    !python -m scripts.analyse cost --dir outputs/experiments --llm-ms 1780
    !python -m scripts.analyse gap  --dir outputs/experiments
    !python -m scripts.collect_results --dir outputs/experiments

## 9. Save to Drive

In [ ]:
import shutil

# Colab sessions are wiped without warning. Copy anything expensive to Drive
# as soon as it exists, not at the end of the notebook.
#
# What is deliberately NOT copied:
#
#   splits_*      contain verbatim post text, needed for training. They
#                 regenerate from the labels in seconds, so persisting them
#                 buys nothing and puts post text on third-party storage.
#   model,        half a gigabyte per training run, ten runs in the grid,
#   checkpoints   and nothing downstream reads them.
#
# The labels file IS copied and is the one thing that must survive: it costs
# hours to regenerate and carries no post text, only post_id and labels.
SKIP = shutil.ignore_patterns("splits_*", "model", "checkpoints", "checkpoint-*")

for src in ["outputs", "data/processed"]:
    dst = WORK / src.replace("/", "_")
    if Path(src).exists():
        shutil.copytree(src, dst, dirs_exist_ok=True, ignore=SKIP)
        print(f"{src} -> {dst}")

# Fail loudly if post text reached Drive anyway, rather than discovering it
# later. Checks for the text field the training splits carry.
leaked = [f for f in WORK.rglob("*.jsonl")
          if any('"text":' in line for line in f.open().readlines()[:5])]
if leaked:
    print("\nWARNING: files on Drive contain post text:")
    for f in leaked:
        print(f"  {f}")

used = sum(f.stat().st_size for f in WORK.rglob("*") if f.is_file())
print(f"\n{used / 1e6:.1f} MB in {WORK}")
